In [ ]:
!pip install -q pydicom opencv-python

In [ ]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print("TensorFlow :", tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset_path = "/content/drive/MyDrive/pfe_fati_chaimae/Dataset_balanced"

classes = ["AD", "MCI", "CN"]

label_map = {
    "AD": 0,
    "MCI": 1,
    "CN": 2
}

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
print("TensorFlow :", tf.__version__)

print("GPU :", tf.config.list_physical_devices('GPU'))

In [ ]:
paths = []
labels = []

for cls in classes:

    class_path = os.path.join(dataset_path, cls)

    print(f"Scanning {cls} ...")

    for patient in os.listdir(class_path):

        patient_path = os.path.join(class_path, patient)

        for root, _, files in os.walk(patient_path):

            for file in files:

                if file.lower().endswith(".dcm"):

                    paths.append(os.path.join(root, file))

                    labels.append(label_map[cls])

print("Nombre d'images :", len(paths))

Création du DataFrame

In [ ]:
df = pd.DataFrame({

    "Path": paths,

    "Label": labels

})

print(df.head())

print()

print(df.shape)

In [ ]:
print(df["Label"].value_counts())

plt.figure(figsize=(6,4))

sns.countplot(
    x=df["Label"]
)

plt.xticks(
    [0,1,2],
    ["AD","MCI","CN"]
)

plt.title("Distribution des classes")

plt.show()

In [ ]:
train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42,

    stratify=df["Label"]

)

print("Train :", len(train_df))

print("Test :", len(test_df))

In [ ]:
train_df = train_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

print(train_df.head())

In [ ]:
def read_dicom(path):

    dcm = pydicom.dcmread(path)

    image = dcm.pixel_array.astype(np.float32)

    image = cv2.resize(image, (224,224))

    image = (image-image.min())/(image.max()-image.min()+1e-8)

    image = np.stack([image,image,image],axis=-1)

    return image

In [ ]:
from tensorflow.keras.utils import Sequence

class DicomDataGenerator(Sequence):

    def __init__(self, dataframe, batch_size=32, shuffle=True, augment=False):

        self.df = dataframe.copy()

        self.batch_size = batch_size

        self.shuffle = shuffle

        self.augment = augment

        self.indexes = np.arange(len(self.df))

        self.on_epoch_end()

    def __len__(self):

        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):

        batch_indexes = self.indexes[
            index*self.batch_size:(index+1)*self.batch_size
        ]

        batch_df = self.df.iloc[batch_indexes]

        images = []

        labels = []

        for _, row in batch_df.iterrows():

            img = read_dicom(row["Path"])

            if self.augment:

                img = self.apply_augmentation(img)

            images.append(img)

            labels.append(row["Label"])

        return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(self.indexes)

    def apply_augmentation(self, image):

        if np.random.rand() < 0.5:
            image = np.fliplr(image)

        if np.random.rand() < 0.3:
            image = np.flipud(image)

        return image

In [ ]:
train_generator = DicomDataGenerator(

    train_df,

    batch_size=32,

    shuffle=True,

    augment=True

)

test_generator = DicomDataGenerator(

    test_df,

    batch_size=32,

    shuffle=False,

    augment=False

)

In [ ]:
images, labels = train_generator[0]

print(images.shape)

print(labels.shape)

print(images.dtype)

print(labels.dtype)

In [ ]:
plt.figure(figsize=(12,6))

for i in range(6):

    plt.subplot(2,3,i+1)

    plt.imshow(images[i])

    plt.title(classes[labels[i]])

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
print("Train batches :", len(train_generator))

print("Test batches :", len(test_generator))

Construction du modèle MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Freeze MobileNetV2
base_model.trainable = False

inputs = Input(shape=(224,224,3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

x = Dropout(0.5)(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.3)(x)

outputs = Dense(3, activation="softmax")(x)

model = Model(inputs, outputs)

model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(

    optimizer=Adam(learning_rate=1e-4),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True

)

checkpoint = ModelCheckpoint(

    "/content/drive/MyDrive/pfe_fati_chaimae/MobileNetV2_best.keras",

    monitor="val_accuracy",

    save_best_only=True

)

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.2,

    patience=3,

    min_lr=1e-6

)

Training Time

In [ ]:
import time

start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=20,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

training_time = time.time() - start_time

print(f"Training Time: {training_time:.2f} seconds")

In [ ]:
history = model.fit(

    train_generator,

    validation_data=test_generator,

    epochs=20,

    callbacks=[

        early_stop,

        checkpoint,

        reduce_lr

    ],

    verbose=1

)

In [ ]:
loss, accuracy = model.evaluate(

    test_generator,

    verbose=1

)

print("Test Loss :", loss)

print("Test Accuracy :", accuracy)

Accuracy & Loss Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(history.history["accuracy"])

plt.plot(history.history["val_accuracy"])

plt.legend(["Train","Validation"])

plt.title("Accuracy")

plt.grid()

plt.subplot(1,2,2)

plt.plot(history.history["loss"])

plt.plot(history.history["val_loss"])

plt.legend(["Train","Validation"])

plt.title("Loss")

plt.grid()

plt.show()

Confusion Matrix

In [ ]:
y_pred = model.predict(test_generator)

y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(

    test_df["Label"],

    y_pred_classes

)

plt.figure(figsize=(6,5))

sns.heatmap(

    cm,

    annot=True,

    cmap="Blues",

    fmt="d",

    xticklabels=["AD","MCI","CN"],

    yticklabels=["AD","MCI","CN"]

)

plt.xlabel("Prediction")

plt.ylabel("True")

plt.show()

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize

# ==========================
# Prediction
# ==========================

y_true = []
y_pred = []
y_prob = []

for images, labels in test_dataset:

    predictions = model.predict(images, verbose=0)

    y_prob.extend(predictions)

    y_pred.extend(np.argmax(predictions, axis=1))

    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# ==========================
# Metrics
# ==========================

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

# ==========================
# AUC
# ==========================

y_true_bin = label_binarize(
    y_true,
    classes=[0,1,2]
)

auc_score = roc_auc_score(
    y_true_bin,
    y_prob,
    average="weighted",
    multi_class="ovr"
)

# ==========================
# Number of Parameters
# ==========================

parameters = model.count_params()

# ==========================
# Results
# ==========================

results = pd.DataFrame({

    "Accuracy":[accuracy],

    "Precision":[precision],

    "Recall":[recall],

    "F1 Score":[f1],

    "AUC":[auc_score],

    "Training Time":[training_time],

    "Parameters":[parameters]

})

print(results)

Classification Report

In [ ]:
print(

    classification_report(

        test_df["Label"],

        y_pred_classes,

        target_names=["AD","MCI","CN"]

    )

)

ROC Curve + AUC

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Prediction probabilities
y_score = model.predict(test_generator)

# True labels
y_true = test_df["Label"].values

# Binarization
n_classes = 3
y_true_bin = label_binarize(y_true, classes=[0,1,2])

# ROC + AUC
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_true_bin[:, i],
        y_score[:, i]
    )

    roc_auc[i] = auc(
        fpr[i],
        tpr[i]
    )

# Plot
plt.figure(figsize=(8,6))

colors = ["blue","green","red"]
class_names = ["AD","MCI","CN"]

for i, color in zip(range(n_classes), colors):

    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f"{class_names[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend(loc="lower right")

plt.grid(True)

plt.show()

Mean AUC

In [ ]:
mean_auc = np.mean(list(roc_auc.values()))

print(f"Mean AUC : {mean_auc:.4f}")

Fine-Tuning

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(

    optimizer=Adam(learning_rate=1e-5),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_finetune = model.fit(

    train_generator,

    validation_data=test_generator,

    epochs=10,

    callbacks=[

        early_stop,

        checkpoint,

        reduce_lr

    ]

)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
for layer in model.layers:
    print(layer.name)

Fonction Grad-CAM

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)

        pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0,1,2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(
        pooled_grads * conv_outputs,
        axis=-1
    )

    heatmap = tf.maximum(heatmap,0)

    heatmap /= tf.reduce_max(heatmap)+1e-8

    return heatmap.numpy()

In [ ]:
# Choisir une image du test set
index = 0

img = read_dicom(test_df.iloc[index]["Path"])

true_label = test_df.iloc[index]["Label"]

img_batch = np.expand_dims(img, axis=0)

# Heatmap
heatmap = make_gradcam_heatmap(
    img_batch,
    model,
    "out_relu"      # changer si nécessaire
)

# Prediction
prediction = model.predict(img_batch, verbose=0)

predicted_label = np.argmax(prediction)

confidence = np.max(prediction)

# Resize heatmap
heatmap = cv2.resize(
    heatmap,
    (IMG_SIZE, IMG_SIZE)
)

heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(
    heatmap,
    cv2.COLORMAP_JET
)

overlay = cv2.addWeighted(
    (img*255).astype(np.uint8),
    0.6,
    heatmap,
    0.4,
    0
)

# Affichage
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Original MRI")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(heatmap)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.show()

print("True Label :", classes[true_label])
print("Prediction :", classes[predicted_label])
print("Confidence :", confidence)

Sauvegarde du modèle

In [ ]:
model.save(

    "/content/drive/MyDrive/pfe_fati_chaimae/MobileNetV2.keras"

)
results.to_csv(

    "/content/drive/MyDrive/pfe_fati_chaimae/MobileNetV2_results.csv",

    index=False

)

print(results)